# MICrONS Option 3 — Time-resolved PCA trajectories

Per-area, per-stimulus trial-averaged trajectories with the time axis preserved. Resolves the Monet2/Trippy collapse seen in Option 2 by keeping within-trial dynamics. See `docs/specs/2026-05-04-option3-trajectories-design.md` for design and `docs/plans/2026-05-04-option3-trajectories-implementation.md` for the implementation plan.

## How to use this notebook

Pipeline is **session-swappable**: change `SESSION` in the configuration block below and restart the kernel.

## Part 0 — Setup

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import microns_eda
import option2_pca_utils
import option3_trajectories_utils as traj_utils

In [ ]:
# === Configuration block — every tunable lives here. ===
SESSION = "7_5"

# Preprocessing (reused from Option 2).
N_FRAMES_TRUNCATE = 75
TREADMILL_OUTLIER_THRESHOLD = 1.0

# PCA / metrics.
N_COMPONENTS = 10
N_PCS_FOR_DISTANCE = 3

# Bootstrap / null.
N_BOOT = 1000
N_SHUFFLES = 100

# Subsampling controls.
N_POPULATION_SUBSAMPLES = 20
N_CLIP_SUBSAMPLES = 20

# Reproducibility.
RANDOM_SEED = 42

# Paths.
DATADIR = Path(os.environ.get("MICRONS_DATADIR", "../neuroscience"))
FIGURES_DIR = Path(f"figures/option3/{SESSION}")
RESULTS_DIR = Path(f"results/option3/{SESSION}")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# 7.5 Hz sampling → ~133 ms/frame, ~10 s total.
def frame_to_ms(f):
    return f * 1000.0 / 7.5

np.random.seed(RANDOM_SEED)

print(f"SESSION       = {SESSION}")
print(f"DATADIR       = {DATADIR}")
print(f"FIGURES_DIR   = {FIGURES_DIR}")
print(f"RESULTS_DIR   = {RESULTS_DIR}")
print(f"75 frames     = {frame_to_ms(75):.1f} ms ≈ {frame_to_ms(75)/1000:.1f} s")

### Sanity check

Verifies the data loads correctly, recomputes `clean_trial_indices`, and checks frame-0 alignment via `stim_times`. All session-specific values are read from data (no hard-coded numbers) so this cell adapts to any session.

In [ ]:
# === Sanity check ===
reader = microns_eda.open_dataset(DATADIR)
sessions = microns_eda.list_sessions(DATADIR)
assert SESSION in sessions, f"session {SESSION!r} not in {sessions}"
print(f"Found {len(sessions)} sessions; using {SESSION!r}.")

meta = microns_eda.get_session_meta(DATADIR, SESSION)
print(f"  n_neurons (read from data) = {meta['n_neurons']}")

# Per-trial treadmill mean → cleaning.
n_trials = meta["n_trials"]
per_trial_tread_means = np.empty(n_trials, dtype=np.float64)
for i in range(n_trials):
    trial = microns_eda.load_trial(reader, DATADIR, SESSION, i)
    per_trial_tread_means[i] = np.nanmean(trial["treadmill"])
clean_trial_indices, running_mask = microns_eda.compute_clean_trial_indices(
    per_trial_tread_means, threshold=TREADMILL_OUTLIER_THRESHOLD,
)

stim_map = microns_eda.build_stim_type_map(reader, SESSION)
stim_types_full = np.array([
    stim_map[h.decode() if isinstance(h, bytes) else h]
    for h in meta["condition_hashes"]
])
labels = stim_types_full[clean_trial_indices]
class_counts = pd.Series(labels).value_counts().to_dict()

print(f"  n_trials (total)            = {n_trials}")
print(f"  n_trials (after running QC) = {len(clean_trial_indices)}")
print(f"  per-class breakdown (clean):")
for k in sorted(class_counts):
    print(f"    {k:8s} {class_counts[k]}")

# Frame-0 alignment check via stim_times.
trial0 = microns_eda.load_trial(reader, DATADIR, SESSION, int(clean_trial_indices[0]))
print(f"  sample trial stim_times shape = {trial0['stim_times'].shape}")
print(f"  sample trial stim_times[:3]   = {trial0['stim_times'][:3]}")
print("  (Frame-0 alignment is implicit in the EDA's truncation; no shift needed.)")

assert len(clean_trial_indices) > 0, "no clean trials remain"
assert len(class_counts) >= 2, "need ≥2 stim classes"
print("Sanity check passed.")

## Part 0.5 — PSTH-like sanity check

Per-area, per-stim, per-frame area-mean response (one number per frame, averaged across neurons). Three curves per area on shared axes, no PCA.

This is the rawest temporal view of stimulus differences. We look here first, before any geometric analysis. **What we're spotting:** does the population mean rise within the first frames after stimulus onset? Does it saturate or decay? Do the three stim classes already separate at the per-frame mean level (Option 2 said yes, this confirms it visually)?

In [ ]:
# We need the preprocessed responses + per-area trajectories for PSTH; build them once here
# (Part 1 will use the same outputs).
print("loading + preprocessing responses...")
responses, trial_boundaries, _ = microns_eda.load_session_responses(
    reader, DATADIR, SESSION
)
responses_pp = option2_pca_utils.preprocess_responses(responses, apply_log=False)
trajectories = traj_utils.build_stim_trajectories(
    responses_pp, trial_boundaries, clean_trial_indices,
    meta["brain_areas"], labels, n_frames=N_FRAMES_TRUNCATE,
)
print(f"trajectories built for areas {sorted(trajectories.keys())}")

# Four-panel PSTH (one per area).
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for area, ax in zip(["V1", "AL", "LM", "RL"], axes.flat):
    if area in trajectories:
        traj_utils.plot_psth_per_stim_area(trajectories, area, ax=ax)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "0_5_psth_per_area.png", dpi=150)
plt.show()

## Part 1 — Build per-area, per-stimulus trajectories

We've already built `trajectories` above for the PSTH plot. Here we save them to disk for later reload (so subsequent runs of the notebook can skip the slow preprocessing step) and run sanity asserts.

In [ ]:
# Sanity asserts on the trajectory shapes.
for area, per_stim in trajectories.items():
    for stim, traj in per_stim.items():
        assert traj.shape[0] == N_FRAMES_TRUNCATE, f"{area}/{stim} wrong n_frames"
        assert not np.isnan(traj).any(), f"{area}/{stim} has NaN"
        assert (traj.std(axis=0) > 0).all(), f"{area}/{stim} has zero-var neurons"
print("trajectory sanity asserts passed.")

# Save to disk for fast reload.
traj_path = RESULTS_DIR / "trajectories.npz"
traj_utils.save_trajectories(trajectories, traj_path)
print(f"saved {traj_path}")

## Part 2 — Fit PCA per area on stacked stimulus trajectories

For each area, vertically stack the 3 stimulus matrices into a `(225, n_neurons_in_area)` matrix and fit PCA. The 3 stimuli share a common coordinate system within an area, so trajectories are directly comparable.

**Note: PC1 in this stacked-matrix PCA is typically dominated by stimulus-common temporal dynamics** (rise at onset, settle later) — visualisations should focus on PC2-3 separation as the substantive stimulus-identity finding, not PC1.

In [ ]:
pca_per_area = traj_utils.fit_trajectory_pca(
    trajectories, n_components=N_COMPONENTS, random_state=RANDOM_SEED,
)
print("Per-area PCA fitted:")
for area in sorted(pca_per_area.keys()):
    pca = pca_per_area[area]["pca"]
    top3 = 100 * pca.explained_variance_ratio_[:3].sum()
    top10 = 100 * pca.explained_variance_ratio_[:10].sum()
    print(f"  {area:4s}: top-3 cum var = {top3:5.1f}%   top-10 cum var = {top10:5.1f}%")

## Part 3 — Visualise V1 trajectories

V1 is our reference area. We plot the three stimulus trajectories in the top-3 PC space — 2-D for the report, 3-D Plotly for the oral presentation. Filled circle marks frame 0, open circle marks the last frame.

**What we're looking for:** do the trajectories peel apart at any point in the trial, or do they trace overlapping paths? PC1 likely captures stimulus-common temporal dynamics (the joint rise/settle); stimulus-specific differences should appear as separation along PC2-3.

**Qualitative shape question:** even when trajectories peel apart, *how* do they peel apart? Parallel curves at different points = stable per-stim representation. Intersecting / diverging curves = dynamic re-coding.

In [ ]:
# 2-D trajectory (matplotlib).
fig, ax = plt.subplots(figsize=(8, 6))
traj_utils.plot_trajectory_2d(
    pca_per_area["V1"]["stim_pcs"], "V1", pca_per_area["V1"]["pca"], ax=ax,
)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "3_v1_trajectory_2d.png", dpi=150)
plt.show()

# 3-D trajectory (Plotly).
fig3d = traj_utils.plot_trajectory_3d_plotly(
    pca_per_area["V1"]["stim_pcs"], "V1", pca_per_area["V1"]["pca"],
)
out_html = FIGURES_DIR / "3_v1_trajectory_3d.html"
fig3d.write_html(str(out_html))
print(f"saved {out_html}")
fig3d.show()

### Interpretation — V1 trajectories

*[Add after running: do the three trajectories peel apart? Where (PC1, PC2, PC3)? What is the qualitative shape — parallel, intersecting, or diverging?]*

## Part 4 — Pairwise distance time courses for V1

For each pair of stimulus trajectories, compute frame-by-frame Euclidean distance. We compute two metrics:

1. **Full-feature distance** (primary): uses all V1 neurons. Honest measure of trajectory separation; not dependent on which directions PCA happens to select.
2. **Top-3 PC distance** (visualisation-aligned check): uses the projection through V1's PCA. Tells us how much of the full separation lives in the dimensions we plotted in Part 3.

If the two metrics agree (both rise together, both reach significance at similar latencies), the visualisation captures the real story. If they disagree (e.g., full-feature separates strongly but top-3 PC doesn't), the class signal lives in lower-variance directions — same situation we saw in Option 2.

In [ ]:
# Build the V1 trial × time × neuron tensor (used by Parts 4-5 + bootstrap + null).
def _build_trial_tensor(responses_pp, trial_boundaries, clean_indices, cols, n_frames):
    out = np.empty((len(clean_indices), n_frames, len(cols)), dtype=np.float64)
    for out_idx, t_idx in enumerate(clean_indices):
        start = int(trial_boundaries[t_idx])
        out[out_idx] = responses_pp[cols, start:start + n_frames].T
    return out

# Decode area assignments once.
brain_areas_str = np.array([
    s.decode("utf-8") if isinstance(s, bytes) else str(s)
    for s in meta["brain_areas"]
])
v1_cols = np.where(brain_areas_str == "V1")[0]
v1_tensor = _build_trial_tensor(
    responses_pp, trial_boundaries, clean_trial_indices, v1_cols, N_FRAMES_TRUNCATE,
)
print(f"V1 trial tensor shape: {v1_tensor.shape}")  # (453, 75, 5485) for 7_5

# Compute distances (full-feature + top-3 PC).
v1_d_full = traj_utils.pairwise_trajectory_distance(
    trajectories["V1"], metric="full",
)
v1_d_top3 = traj_utils.pairwise_trajectory_distance(
    trajectories["V1"], metric="top_pcs",
    pca=pca_per_area["V1"]["pca"], n_pcs=N_PCS_FOR_DISTANCE,
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
traj_utils.plot_pairwise_distance_time_course(
    v1_d_full, None, "V1", "Euclidean (full features)", ax=axes[0],
)
traj_utils.plot_pairwise_distance_time_course(
    v1_d_top3, None, "V1", "Euclidean (top-3 PCs)", ax=axes[1],
)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "4_v1_distance_two_metrics.png", dpi=150)
plt.show()

### Interpretation — V1 distance metrics

*[Add after running: do the two metrics agree? If yes, the visualisation in Part 3 captures the real story. If no, note where the disagreement lives — which pair, which time window.]*

## Part 5 — Bootstrap envelopes + shuffle null on V1

Two statistical hygiene checks:

- **Bootstrap envelope** (B=1000, class-stratified resampling): for each frame and each pair, the 95% CI of the distance under repeated trial-resampling. Shows whether observed wiggles reflect biology or sampling noise.
- **Shuffle null** (n=100, max-distance statistic): if stim labels were random, what max pairwise distance would we expect? The observed max compared to this null gives an empirical p-value per pair.

Pre-registered decision: a pair is **reliably separated** if (a) p < 0.05 against the shuffle null AND (b) the bootstrap *lower* envelope clears the shuffle null *upper* (95th percentile) at some frame.

In [ ]:
print("computing V1 bootstrap envelopes (B=1000)...")
v1_env_full = traj_utils.bootstrap_distance_envelope(
    v1_tensor, labels, metric="full",
    n_boot=N_BOOT, seed=RANDOM_SEED,
)
v1_env_top3 = traj_utils.bootstrap_distance_envelope(
    v1_tensor, labels, metric="top_pcs",
    pca=pca_per_area["V1"]["pca"], n_pcs=N_PCS_FOR_DISTANCE,
    n_boot=N_BOOT, seed=RANDOM_SEED + 1,
)

print("computing V1 shuffle null (n=100)...")
v1_null_full = traj_utils.shuffle_null_max_distance(
    v1_tensor, labels, metric="full",
    n_shuffles=N_SHUFFLES, seed=RANDOM_SEED + 2,
)
v1_null_top3 = traj_utils.shuffle_null_max_distance(
    v1_tensor, labels, metric="top_pcs",
    pca=pca_per_area["V1"]["pca"], n_pcs=N_PCS_FOR_DISTANCE,
    n_shuffles=N_SHUFFLES, seed=RANDOM_SEED + 3,
)

In [ ]:
# Per-pair p-values + onset latencies.
def _summarize_v1(distances, envelopes, nulls):
    rows = []
    for pair, d in distances.items():
        observed_max = float(d.max())
        null_p95 = float(np.percentile(nulls[pair], 95))
        p = float((1 + (nulls[pair] >= observed_max).sum()) / (1 + len(nulls[pair])))
        latency = traj_utils.compute_onset_latency(
            envelopes[pair]["lower"], nulls[pair], alpha=0.05
        )
        rows.append({
            "pair": "/".join(sorted(pair)),
            "max_observed": observed_max,
            "null_p95": null_p95,
            "p_value": p,
            "onset_frame": latency,
            "onset_ms": frame_to_ms(latency) if latency is not None else None,
        })
    return pd.DataFrame(rows)

print("=== V1 full-feature ===")
print(_summarize_v1(v1_d_full, v1_env_full, v1_null_full).to_string(index=False))
print("\n=== V1 top-3 PCs ===")
print(_summarize_v1(v1_d_top3, v1_env_top3, v1_null_top3).to_string(index=False))

In [ ]:
# Two-panel figure with envelopes + null thresholds.
v1_null_p95_full = {p: float(np.percentile(v1_null_full[p], 95)) for p in v1_null_full}
v1_null_p95_top3 = {p: float(np.percentile(v1_null_top3[p], 95)) for p in v1_null_top3}

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
traj_utils.plot_pairwise_distance_time_course(
    v1_d_full, v1_env_full, "V1", "Euclidean (full features)",
    null_p95=v1_null_p95_full, ax=axes[0],
)
traj_utils.plot_pairwise_distance_time_course(
    v1_d_top3, v1_env_top3, "V1", "Euclidean (top-3 PCs)",
    null_p95=v1_null_p95_top3, ax=axes[1],
)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "5_v1_distance_with_envelopes.png", dpi=150)
plt.show()

### Interpretation — V1 statistical results

*[Add after running: which pairs reach reliable separation? At what onset latencies (in ms)? Does the Monet2/Trippy pair separate, or does the Option 2 collapse persist?]*

## Part 6 — Repeat on AL, LM, RL

Apply Parts 3-5 to the higher visual areas: 2-D + 3-D scatters, full-feature + top-3 PC distance time courses with bootstrap envelopes and shuffle nulls.

In [ ]:
all_area_results = {
    "V1": {
        "tensor": v1_tensor,
        "trajectories": trajectories["V1"],
        "pca": pca_per_area["V1"]["pca"],
        "stim_pcs": pca_per_area["V1"]["stim_pcs"],
        "distances_full": v1_d_full,
        "distances_top3": v1_d_top3,
        "envelopes_full": v1_env_full,
        "envelopes_top3": v1_env_top3,
        "null_full": v1_null_full,
        "null_top3": v1_null_top3,
        "n_neurons": v1_tensor.shape[2],
    },
}

for area in ["AL", "LM", "RL"]:
    print(f"\n=== {area} ===")
    cols = np.where(brain_areas_str == area)[0]
    tensor = _build_trial_tensor(
        responses_pp, trial_boundaries, clean_trial_indices, cols, N_FRAMES_TRUNCATE,
    )
    print(f"  trial tensor shape: {tensor.shape}")

    pca_a = pca_per_area[area]["pca"]
    stim_pcs_a = pca_per_area[area]["stim_pcs"]
    d_full = traj_utils.pairwise_trajectory_distance(
        trajectories[area], metric="full",
    )
    d_top3 = traj_utils.pairwise_trajectory_distance(
        trajectories[area], metric="top_pcs",
        pca=pca_a, n_pcs=N_PCS_FOR_DISTANCE,
    )
    print(f"  bootstrap envelopes (B={N_BOOT})...")
    env_full = traj_utils.bootstrap_distance_envelope(
        tensor, labels, metric="full",
        n_boot=N_BOOT, seed=RANDOM_SEED + 10 + ord(area[0]),
    )
    env_top3 = traj_utils.bootstrap_distance_envelope(
        tensor, labels, metric="top_pcs",
        pca=pca_a, n_pcs=N_PCS_FOR_DISTANCE,
        n_boot=N_BOOT, seed=RANDOM_SEED + 20 + ord(area[0]),
    )
    print(f"  shuffle null (n={N_SHUFFLES})...")
    null_full = traj_utils.shuffle_null_max_distance(
        tensor, labels, metric="full",
        n_shuffles=N_SHUFFLES, seed=RANDOM_SEED + 30 + ord(area[0]),
    )
    null_top3 = traj_utils.shuffle_null_max_distance(
        tensor, labels, metric="top_pcs",
        pca=pca_a, n_pcs=N_PCS_FOR_DISTANCE,
        n_shuffles=N_SHUFFLES, seed=RANDOM_SEED + 40 + ord(area[0]),
    )
    all_area_results[area] = {
        "tensor": tensor,
        "trajectories": trajectories[area],
        "pca": pca_a,
        "stim_pcs": stim_pcs_a,
        "distances_full": d_full,
        "distances_top3": d_top3,
        "envelopes_full": env_full,
        "envelopes_top3": env_top3,
        "null_full": null_full,
        "null_top3": null_top3,
        "n_neurons": tensor.shape[2],
    }

In [ ]:
# Per-area scatters (2-D + 3-D Plotly).
for area in ["AL", "LM", "RL"]:
    res = all_area_results[area]
    fig, ax = plt.subplots(figsize=(8, 6))
    traj_utils.plot_trajectory_2d(res["stim_pcs"], area, res["pca"], ax=ax)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / f"6_{area.lower()}_trajectory_2d.png", dpi=150)
    plt.show()

    fig3d = traj_utils.plot_trajectory_3d_plotly(res["stim_pcs"], area, res["pca"])
    fig3d.write_html(str(FIGURES_DIR / f"6_{area.lower()}_trajectory_3d.html"))

## Part 7 — Cross-area Monet2↔Trippy summary (headline)

The headline figure: Monet2↔Trippy distance time course for all four areas on shared axes, with bootstrap envelopes. Two panels: full-feature distance (primary) and top-3 PC distance (visualisation check).

In [ ]:
m2t_pair = frozenset({"Monet2", "Trippy"})
m2t_full = {area: res["distances_full"][m2t_pair] for area, res in all_area_results.items()}
m2t_env_full = {area: res["envelopes_full"][m2t_pair] for area, res in all_area_results.items()}
m2t_top3 = {area: res["distances_top3"][m2t_pair] for area, res in all_area_results.items()}
m2t_env_top3 = {area: res["envelopes_top3"][m2t_pair] for area, res in all_area_results.items()}

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
traj_utils.plot_cross_area_monet2_trippy(
    m2t_full, m2t_env_full, "Euclidean (full features)", ax=axes[0],
)
traj_utils.plot_cross_area_monet2_trippy(
    m2t_top3, m2t_env_top3, "Euclidean (top-3 PCs)", ax=axes[1],
)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "7_cross_area_monet2_trippy.png", dpi=150)
plt.show()

In [ ]:
# Per-area, per-pair, per-metric summary CSV.
def _build_metric_rows(results, analysis_label):
    rows = []
    for area, res in results.items():
        for metric_label, dist_key, env_key, null_key in [
            ("full", "distances_full", "envelopes_full", "null_full"),
            ("top3_pc", "distances_top3", "envelopes_top3", "null_top3"),
        ]:
            for pair in res[dist_key]:
                d = res[dist_key][pair]
                env = res[env_key][pair]
                null = res[null_key][pair]
                obs_max = float(d.max())
                p95 = float(np.percentile(null, 95))
                p = float((1 + (null >= obs_max).sum()) / (1 + len(null)))
                lat = traj_utils.compute_onset_latency(env["lower"], null, alpha=0.05)
                rows.append({
                    "area": area,
                    "n_neurons": res["n_neurons"],
                    "pair": "/".join(sorted(pair)),
                    "metric": metric_label,
                    "max_distance_observed": obs_max,
                    "max_distance_null_p95": p95,
                    "empirical_p_value": p,
                    "onset_frame": lat,
                    "onset_ms": frame_to_ms(lat) if lat is not None else None,
                    "analysis": analysis_label,
                })
    return rows

results_df = pd.DataFrame(_build_metric_rows(all_area_results, "all_neurons"))
csv_path = RESULTS_DIR / "trajectory_metrics.csv"
results_df.to_csv(csv_path, index=False)
print(f"wrote {csv_path}  ({len(results_df)} rows)")
results_df

### Interpretation — cross-area headline

*[Add after running: where (if anywhere) does Monet2↔Trippy reach reliable separation? V1 first, higher areas first, or simultaneously? At what onset latencies in ms?]*

## Part 8 — Equal-population control

V1 has ~13× more neurons than AL. Distances scale with population size. We subsample V1, LM, RL down to AL's neuron count (414), 20 random subsamples each, and check whether the cross-area Monet2↔Trippy ranking from Part 7 survives population matching.

For compute reasons, **bootstrap envelopes and shuffle nulls are NOT recomputed per subsample** — the cross-area ranking question is answered by comparing the median ± IQR of observed distances across the 20 subsamples.

In [ ]:
n_match = all_area_results["AL"]["n_neurons"]
print(f"n_match (AL n_neurons, read from data) = {n_match}")

matched_results: dict[str, list[dict]] = {"AL": [{
    "trajectories": all_area_results["AL"]["trajectories"],
    "distances_full": all_area_results["AL"]["distances_full"],
    "distances_top3_pc": all_area_results["AL"]["distances_top3"],
}]}

for area in ["V1", "LM", "RL"]:
    print(f"\n=== {area} (population matched, {N_POPULATION_SUBSAMPLES} subsamples) ===")
    matched_results[area] = traj_utils.subsample_population_run_pipeline(
        area_name=area,
        trial_tensor=all_area_results[area]["tensor"],
        labels=labels,
        n_match=n_match,
        n_subsamples=N_POPULATION_SUBSAMPLES,
        n_components=N_COMPONENTS,
        seed=RANDOM_SEED + ord(area[0]),
    )
    print(f"  done — {len(matched_results[area])} subsamples")

In [ ]:
# Cross-area Monet2↔Trippy under matching: median + IQR over subsamples.
m2t_pair = frozenset({"Monet2", "Trippy"})
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
for ax, metric_key, metric_label in [
    (axes[0], "distances_full", "Euclidean (full features)"),
    (axes[1], "distances_top3_pc", "Euclidean (top-3 PCs)"),
]:
    for area in ["V1", "AL", "LM", "RL"]:
        per_sub = matched_results[area]
        # Stack distances across subsamples.
        arr = np.array([d[metric_key][m2t_pair] for d in per_sub])
        median = np.median(arr, axis=0)
        x = np.arange(len(median))
        palette = {"V1": "#0072B2", "AL": "#E69F00", "LM": "#009E73", "RL": "#CC79A7"}
        ax.plot(x, median, "-", color=palette[area], linewidth=2.2, label=area)
        if arr.shape[0] > 1:
            q25 = np.quantile(arr, 0.25, axis=0)
            q75 = np.quantile(arr, 0.75, axis=0)
            ax.fill_between(x, q25, q75, color=palette[area], alpha=0.2)
    ax.set_xlabel("Frame")
    ax.set_ylabel(metric_label)
    ax.set_title(f"Monet2↔Trippy at matched population n={n_match} ({metric_label})")
    ax.legend(loc="best", fontsize=9)
    ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "8_population_matched_monet2_trippy.png", dpi=150)
plt.show()

In [ ]:
# Append matched-population rows to the per-session CSV.
matched_rows = []
for area, sub_list in matched_results.items():
    n_n = n_match
    for metric_label, key in [("full", "distances_full"), ("top3_pc", "distances_top3_pc")]:
        # Aggregate per pair across subsamples.
        pair_keys = list(sub_list[0][key].keys())
        for pair in pair_keys:
            arr = np.array([d[key][pair] for d in sub_list])
            obs_max_per_sub = arr.max(axis=1)
            matched_rows.append({
                "area": area,
                "n_neurons": n_n,
                "pair": "/".join(sorted(pair)),
                "metric": metric_label,
                "max_distance_observed": float(np.median(obs_max_per_sub)),
                "max_distance_null_p95": np.nan,
                "empirical_p_value": np.nan,
                "onset_frame": None,
                "onset_ms": None,
                "analysis": "equal_population",
            })
existing = pd.read_csv(csv_path) if csv_path.exists() else pd.DataFrame()
existing = existing[existing["analysis"] != "equal_population"] if len(existing) else existing
combined = pd.concat([existing, pd.DataFrame(matched_rows)], ignore_index=True)
combined.to_csv(csv_path, index=False)
print(f"wrote {csv_path}  ({len(combined)} rows total)")

### Interpretation — equal-population

*[Add after running: does the area ranking from Part 7 persist at matched population? If V1's lead disappears, the original lead was a population-size artefact. If it persists, the original difference reflects information content.]*

## Part 9 — Clip-trial subsampling sanity check (V1 only)

Clip has 377 trials, Monet2 and Trippy have only 38 each. Clip's trajectory is therefore much smoother than the synthetics' — could the apparent Clip↔synthetic distances reflect this asymmetry rather than real geometric separation?

We subsample Clip down to 38 trials, repeat the V1 trajectory + distance computation 20 times, and compare to the all-Clip version.

In [ ]:
print(f"V1 Clip-subsampling: {N_CLIP_SUBSAMPLES} subsamples to 38 trials each...")
clip_subsamples = traj_utils.subsample_clip_trials_run_pipeline(
    trial_tensor=v1_tensor,
    labels=labels,
    n_clip_target=38,  # match minority count
    n_subsamples=N_CLIP_SUBSAMPLES,
    n_components=N_COMPONENTS,
    seed=RANDOM_SEED + 100,
)
print(f"  done — {len(clip_subsamples)} subsamples")

In [ ]:
# Compare full-Clip vs subsampled-Clip distance time courses.
fig, ax = plt.subplots(figsize=(10, 5.5))
sub_distances_full = [s["distances_full"] for s in clip_subsamples]
traj_utils.plot_clip_subsampling_comparison(
    full_distances=v1_d_full,
    subsampled_distances_list=sub_distances_full,
    metric_label="Euclidean (full features)",
    ax=ax,
)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "9_v1_clip_subsampling.png", dpi=150)
plt.show()

In [ ]:
# Append clip-matched rows to the CSV.
clip_rows = []
n_n = v1_tensor.shape[2]
for metric_label, key in [("full", "distances_full"), ("top3_pc", "distances_top3_pc")]:
    pair_keys = list(clip_subsamples[0][key].keys())
    for pair in pair_keys:
        arr = np.array([d[key][pair] for d in clip_subsamples])
        obs_max_per_sub = arr.max(axis=1)
        clip_rows.append({
            "area": "V1",
            "n_neurons": n_n,
            "pair": "/".join(sorted(pair)),
            "metric": metric_label,
            "max_distance_observed": float(np.median(obs_max_per_sub)),
            "max_distance_null_p95": np.nan,
            "empirical_p_value": np.nan,
            "onset_frame": None,
            "onset_ms": None,
            "analysis": "clip_matched",
        })
existing = pd.read_csv(csv_path)
existing = existing[existing["analysis"] != "clip_matched"]
combined = pd.concat([existing, pd.DataFrame(clip_rows)], ignore_index=True)
combined.to_csv(csv_path, index=False)
print(f"wrote {csv_path}  ({len(combined)} rows total)")

### Interpretation — Clip subsampling

*[Add after running: do Clip↔Monet2 / Clip↔Trippy distances drop substantially when Clip is matched to 38 trials? If yes, part of the original separation was a sample-size artefact; report both versions.]*

## Part 10 — Cross-session aggregation

Reads `results/option3/{session}/trajectory_metrics.csv` for every session present and produces a comparison plot. With one session this is a single-session view; with ≥2 sessions, a cross-session comparison.

To replicate on another session: change `SESSION` in the configuration block at the top, restart the kernel, and re-run the notebook. Outputs go to per-session subdirectories.

In [ ]:
results_root = RESULTS_DIR.parent
session_dfs = []
for session_dir in sorted(results_root.iterdir()):
    if not session_dir.is_dir():
        continue
    csv = session_dir / "trajectory_metrics.csv"
    if csv.exists():
        df = pd.read_csv(csv)
        df["session"] = session_dir.name
        session_dfs.append(df)

if session_dfs:
    cross_df = pd.concat(session_dfs, ignore_index=True)
    cross_df = cross_df[cross_df["analysis"] == "all_neurons"]
    cross_csv = results_root / "cross_session_summary.csv"
    cross_df.to_csv(cross_csv, index=False)
    print(f"found {cross_df['session'].nunique()} session(s) — wrote {cross_csv}")

    # Headline plot: max Monet2↔Trippy distance per area, per session.
    sub = cross_df[
        (cross_df["pair"] == "Monet2/Trippy") &
        (cross_df["metric"] == "full")
    ]
    if len(sub) > 0:
        fig, ax = plt.subplots(figsize=(9, 4.5))
        sessions = sorted(sub["session"].unique())
        areas = sorted(sub["area"].unique())
        x = np.arange(len(sessions))
        bw = 0.8 / max(len(areas), 1)
        palette = {"V1": "#0072B2", "AL": "#E69F00", "LM": "#009E73", "RL": "#CC79A7"}
        for i, area in enumerate(areas):
            vals = []
            for s in sessions:
                row = sub[(sub["session"] == s) & (sub["area"] == area)]
                vals.append(row["max_distance_observed"].iloc[0] if len(row) else np.nan)
            offset = (i - (len(areas) - 1) / 2) * bw
            ax.bar(x + offset, vals, width=bw, color=palette[area],
                   edgecolor="black", linewidth=0.5, label=area)
        ax.set_xticks(x)
        ax.set_xticklabels(sessions)
        ax.set_xlabel("Session")
        ax.set_ylabel("Max Monet2↔Trippy distance (full features)")
        ax.set_title("Cross-session Monet2↔Trippy max distance by area")
        ax.legend(loc="best", fontsize=8, title="Area")
        ax.grid(True, alpha=0.3, axis="y")
        fig.tight_layout()
        fig.savefig(FIGURES_DIR / "10_cross_session.png", dpi=150)
        plt.show()
else:
    print("no per-session CSVs found; cross-session aggregation is a no-op for now")

## Part 11 — Takeaways

In [ ]:
# Headline numbers per area + the all-neurons + matched-population deltas.
for area in ["V1", "AL", "LM", "RL"]:
    res = all_area_results[area]
    m2t = res["distances_full"][m2t_pair]
    null_p95 = float(np.percentile(res["null_full"][m2t_pair], 95))
    p = float(
        (1 + (res["null_full"][m2t_pair] >= m2t.max()).sum())
        / (1 + len(res["null_full"][m2t_pair]))
    )
    lat = traj_utils.compute_onset_latency(
        res["envelopes_full"][m2t_pair]["lower"], res["null_full"][m2t_pair],
        alpha=0.05,
    )
    sig = "✓" if p < 0.05 else "✗"
    lat_ms = f"{frame_to_ms(lat):.0f} ms" if lat is not None else "n/a"
    print(f"{area:5s}  n={res['n_neurons']:5d}  Monet2↔Trippy max={m2t.max():.3f}  "
          f"null95={null_p95:.3f}  p={p:.3f} {sig}  onset={lat_ms}")

summary_path = RESULTS_DIR / "README.md"
with open(summary_path, "w") as f:
    f.write(f"# Option 3 — session {SESSION} headline numbers\n\n")
    f.write(f"Preprocessing: detrend → z-score; truncate to {N_FRAMES_TRUNCATE} frames; "
            f"running-outlier filter (|treadmill| > {TREADMILL_OUTLIER_THRESHOLD}) → "
            f"{len(clean_trial_indices)} clean trials.\n")
    f.write(f"Class composition (clean): {dict(pd.Series(labels).value_counts())}.\n\n")
    f.write("## Headline: Monet2↔Trippy by area (full-feature metric)\n\n")
    f.write("| Area | n_neurons | max distance | null p95 | p-value | onset (ms) |\n")
    f.write("|---|---|---|---|---|---|\n")
    for area in ["V1", "AL", "LM", "RL"]:
        res = all_area_results[area]
        m2t = res["distances_full"][m2t_pair]
        null_p95 = float(np.percentile(res["null_full"][m2t_pair], 95))
        p = float(
            (1 + (res["null_full"][m2t_pair] >= m2t.max()).sum())
            / (1 + len(res["null_full"][m2t_pair]))
        )
        lat = traj_utils.compute_onset_latency(
            res["envelopes_full"][m2t_pair]["lower"], res["null_full"][m2t_pair],
            alpha=0.05,
        )
        lat_ms = f"{frame_to_ms(lat):.0f}" if lat is not None else "n/a"
        f.write(f"| {area} | {res['n_neurons']} | {m2t.max():.3f} | "
                f"{null_p95:.3f} | {p:.3f} | {lat_ms} |\n")
print(f"\nwrote {summary_path}")

### Headline conclusions

*[Fill in after running the notebook end-to-end:*

- *Monet2↔Trippy verdict per area: who reaches reliable separation, at what onset latency?*
- *Does the equal-population control change the cross-area ranking?*
- *Does the Clip-subsampling control change the V1 picture?*
- *What is the qualitative shape of the V1 trajectories (parallel curves at different points vs intersecting/diverging)?*
- *Implications for CEBRA: where the linear time-resolved analysis succeeded vs. failed.]*